In [1]:
from pandas.core import sample

from foxplainer.explainer import FoX
from foxplainer.explainer import FoX

import sklearn


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report
import pickle
import pandas as pd

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
df = pd.read_csv('../dog_data/dog_adoption_master.csv')

In [3]:
y = df['returned']
y_extra = df[['return_reason','days_to_return']]
X = df.drop(['returned','return_reason','days_to_return','adoption_id'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=14)


X_train_enc = pd.get_dummies(X_train, drop_first=False)
X_test_enc = pd.get_dummies(X_test, drop_first=False)

print('Training set shape: ', np.shape(X_train_enc))
print(f'- Returned:\t {len(y_train[y_train==1])}')
print(f'- Kept: {len(y_train[y_train==0])}')
print('Test set shape: ', np.shape(X_test_enc))
print(f'- Returned:\t {len(y_test[y_test==1])}')
print(f'- Kept: {len(y_test[y_test==0])}')

Training set shape:  (31500, 44)
- Returned:	 4747
- Kept: 26753
Test set shape:  (10500, 44)
- Returned:	 1550
- Kept: 8950


In [4]:
X_train_enc.to_csv('./X_train_dog.csv')

In [4]:
LR = LogisticRegression()
LR.fit(X_train_enc, y_train)

/Users/lkiern/XAI_Framework/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [5]:
preds = LR.predict(X_test_enc)
print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.86      0.98      0.92      8950
           1       0.50      0.10      0.17      1550

    accuracy                           0.85     10500
   macro avg       0.68      0.54      0.54     10500
weighted avg       0.81      0.85      0.81     10500



In [6]:
pickle.dump(LR, open("./models/LR_dog_model.pkl", "wb"))

In [7]:
feature_bounds = {
    i: (float(X_train_enc.iloc[:, i].min()), float(X_train_enc.iloc[:, i].max()))
    for i in range(X_train_enc.shape[1])
}

In [8]:
feature_names = X_train_enc.columns

In [9]:
X_test_enc.iloc[5]

age_years                        7.1
weight_kg                       29.0
days_in_shelter                   14
previously_returned                1
neutered                           1
aggression_score                 1.8
anxiety_separation               5.0
reactivity_to_dogs               5.1
energy_level                     8.7
training_level                   8.1
house_trained                      0
first_time_owner                   0
household_has_kids                 1
household_has_pets                 0
has_yard                           1
hours_alone_per_day              9.7
adopter_activity_level           6.8
visits_before_adoption             1
met_resident_pets                  0
adoption_counseling                0
expectation_score                7.7
energy_mismatch                  1.9
size_home_mismatch                 0
size_large                      True
size_medium                    False
size_small                     False
size_xlarge                    False
b

In [10]:
from foxplainer.explainer import FoX
fx = FoX(global_model_name="LR",
        model_path="./models/LR_dog_model.pkl",
         feature_bounds=feature_bounds,
         time_limit=60,
         feature_names=list(feature_names),
         stakeholder_name='developer')
fx.explain(in_jupyter=True, sample=X_test_enc.iloc[5])


Explaining the logistic regression model...

Entering Feature Attribution
997
min lits: 11, max lits: 20, avg: 14.47


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

### Random Forest

In [11]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

In [12]:
RF = RandomForestClassifier(n_estimators=30, random_state=42)

RF.fit(X_train_enc, y_train)

pickle.dump(RF, open("./models/dog_RF_30estimators_model.pkl", "wb"))


In [13]:
fx = FoX(global_model_name="RF",
        model_path="./models/dog_RF_30estimators_model.pkl",
         feature_bounds=feature_bounds,
         time_limit=60,
         feature_names=list(feature_names),
         stakeholder_name='developer')
fx.explain(in_jupyter=True, sample=X_test_enc.iloc[7])


Explaining the random forest model...

time taken for unit-size MCSes: 0.02 seconds
exiting enumeration
Entering Feature Attribution
No abductive explanations found — FFA cannot be computed. Try increasing the time_limit.


Accordion(children=(Tab(children=(HTML(value=''), HTML(value='\n                \n            <!DOCTYPE HTML>\…

In [14]:
fx = FoX(global_model_name="RF",
        model_path="./models/dog_RF_30estimators_model.pkl",
         feature_bounds=feature_bounds,
         time_limit=120,
         feature_names=list(feature_names),
         stakeholder_name='developer')
fx.explain(in_jupyter=True, sample=X_test_enc.iloc[7])


Explaining the random forest model...

time taken for unit-size MCSes: 0.02 seconds
exiting enumeration
Entering Feature Attribution
2
min lits: 21, max lits: 21, avg: 21.00


Accordion(children=(Tab(children=(HTML(value='\n                \n            <!DOCTYPE HTML>\n            <ht…

In [14]:
fx = FoX(global_model_name="RF",
        model_path="./models/dog_RF_30estimators_model.pkl",
         feature_bounds=feature_bounds,
         time_limit=120,
         feature_names=list(feature_names),
         stakeholder_name='developer')
fx.explain(in_jupyter=True, sample=X_test_enc.iloc[7], prediction_label="Rescue Failed")